### Celda 1: Configuración del entorno y carga de dependencias

Esta celda realiza la configuración inicial del notebook:

- Ajusta el `PYTHONPATH` para incluir la raíz del proyecto, permitiendo importar módulos internos.
- Carga variables de entorno desde el archivo `.env.local`.
- Importa librerías principales para:
  - Manipulación de datos: `pandas`, `numpy`
  - Visualización: `matplotlib`, `seaborn`
  - Análisis y modelado: `sklearn`, `scipy`, `statsmodels`
  - Conexión a base de datos: `psycopg2`
- Verifica versiones de librerías clave y muestra información del entorno.

**Propósito:** asegurar que el entorno está correctamente configurado antes de iniciar el análisis o procesamiento de datos.

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env.local"))

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import scipy
import statsmodels.api as sm

import psycopg2

print("Python:", sys.executable)
print("Proyecto:", PROJECT_ROOT)
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)

print("\nEntorno listo 🚀")

### Celda 2: Conexión a la base de datos

Esta celda define y prueba la conexión a la base de datos PostgreSQL:

- Implementa la función `get_connection()` para crear una conexión usando `psycopg2`.
- Utiliza variables de entorno para los parámetros de conexión (`host`, `port`, `dbname`, `user`, `password`), con valores por defecto.
- Ejecuta una prueba de conexión dentro de un bloque `try-except`:
  - Si es exitosa, imprime confirmación.
  - Si falla, captura y muestra el error.

**Propósito:** validar que la conexión a la base de datos está correctamente configurada antes de realizar operaciones ETL.

In [ ]:
def get_connection():
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "exchange_db"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD", "postgres")
    )
    return conn

try:
    conn = get_connection()
    print("Conexión exitosa")
    conn.close()
except Exception as e:
    print("Error de conexión")
    print(e)

### Celda 3: Extracción y carga de tasas de cambio (ETL - Extract & Load)

Esta celda implementa la lógica principal para obtener y almacenar tasas de cambio:

- Define `get_or_create_currency()`:
  - Busca una moneda por su código en la tabla `currencies`.
  - Si no existe, la crea y retorna su `id`.
  - Evita duplicados (patrón tipo *upsert manual*).

- Define `fetch_and_store_rates()`:
  - Consume una API externa de tasas de cambio usando `requests`.
  - Extrae la moneda base (`base`) y las tasas (`rates`).
  - Abre conexión a la base de datos.
  - Inserta la moneda base si no existe.
  - Itera sobre cada moneda destino:
    - Garantiza su existencia en la tabla `currencies`.
    - Inserta la tasa en `exchange_rates` junto con timestamp actual.

- Realiza commit de la transacción y cierra la conexión.

**Propósito:** poblar la base de datos con tasas de cambio actualizadas desde una fuente externa.

**Nota técnica:** esta celda combina responsabilidades de extracción y carga, representando un flujo ETL básico.

In [6]:
import requests
from datetime import datetime

def get_or_create_currency(cursor, code):
    cursor.execute("SELECT id FROM currencies WHERE code = %s;", (code,))
    result = cursor.fetchone()
    
    if result:
        return result[0]
    
    cursor.execute(
        "INSERT INTO currencies (code) VALUES (%s) RETURNING id;",
        (code,)
    )
    return cursor.fetchone()[0]


def fetch_and_store_rates():
    url = "https://api.exchangerate-api.com/v4/latest/USD"
    
    response = requests.get(url)
    data = response.json()
    
    base_currency = data["base"]
    rates = data["rates"]
    
    conn = get_connection()
    cursor = conn.cursor()
    
    base_id = get_or_create_currency(cursor, base_currency)
    
    inserted = 0
    
    for target_code, rate in rates.items():
        target_id = get_or_create_currency(cursor, target_code)
        
        cursor.execute("""
            INSERT INTO exchange_rates (base_currency_id, target_currency_id, rate, timestamp)
            VALUES (%s, %s, %s, %s);
        """, (
            base_id,
            target_id,
            rate,
            datetime.utcnow()
        ))
        
        inserted += 1
    
    conn.commit()
    cursor.close()
    conn.close()
    
    print(f"Registros insertados: {inserted} 🚀")


fetch_and_store_rates()

Registros insertados: 166 🚀


### Celda 4: Carga de datos desde la base de datos (ETL - Extract)

Esta celda se encarga de extraer los datos almacenados en la base de datos hacia un DataFrame:

- Define la función `load_exchange_rates()`:
  - Ejecuta una consulta SQL sobre la tabla `exchange_rates`.
  - Ordena los registros por `timestamp` para análisis temporal.
  - Utiliza `pandas.read_sql()` para cargar directamente los datos en un DataFrame.
  - Cierra la conexión a la base de datos.

- Ejecuta la función y almacena el resultado en `df`.

- Realiza validaciones básicas:
  - Muestra el número de filas y columnas.
  - Lista los nombres de las columnas.
  - Visualiza las primeras filas con `df.head()`.

**Propósito:** obtener los datos persistidos para su posterior análisis, transformación o visualización.

**Nota técnica:** este paso representa la fase de *Extract* dentro del flujo ETL, desacoplando la lectura de datos del proceso de carga.

In [ ]:
def load_exchange_rates():
    query = """
    SELECT 
        id,
        base_currency_id,
        target_currency_id,
        rate,
        timestamp
    FROM exchange_rates
    ORDER BY timestamp;
    """
    
    conn = get_connection()
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    return df

df = load_exchange_rates()

print("Filas:", len(df))
print("Columnas:", df.columns.tolist())
df.head()

### Celda 5: Carga de datos con información descriptiva (JOIN)

Esta celda mejora la extracción de datos incorporando información más legible:

- Define la función `load_exchange_rates_with_codes()`:
  - Ejecuta una consulta SQL con `JOIN` entre `exchange_rates` y `currencies`.
  - Reemplaza los IDs de monedas por sus códigos (`base_currency`, `target_currency`).
  - Ordena los datos por `timestamp` para mantener consistencia temporal.
  - Carga el resultado en un DataFrame con `pandas.read_sql()`.

- Ejecuta la función y almacena el resultado en `df`.

- Muestra una vista previa con `df.head()`.

**Propósito:** obtener datos enriquecidos y más interpretables para análisis y visualización.

**Nota técnica:** este enfoque aplica una desnormalización controlada, útil para capas analíticas sin afectar el modelo relacional original.

In [ ]:
def load_exchange_rates_with_codes():
    query = """
    SELECT 
        er.id,
        c1.code AS base_currency,
        c2.code AS target_currency,
        er.rate,
        er.timestamp
    FROM exchange_rates er
    JOIN currencies c1 ON er.base_currency_id = c1.id
    JOIN currencies c2 ON er.target_currency_id = c2.id
    ORDER BY er.timestamp;
    """
    
    conn = get_connection()
    df = pd.read_sql(query, conn)
    conn.close()
    
    return df

df = load_exchange_rates_with_codes()

df.head()

### Celda 6: Filtrado de datos específicos (USD a COP)

Esta celda realiza un filtrado sobre el DataFrame para enfocarse en una conversión específica:

- Filtra los registros donde:
  - `base_currency` es `"USD"`
  - `target_currency` es `"COP"`

- Guarda el resultado en `df_usd_cop`.

- Muestra el DataFrame filtrado.

**Propósito:** aislar una serie de tiempo específica (USD → COP) para análisis más detallado.

**Nota técnica:** este tipo de filtrado es clave en análisis exploratorio (EDA), permitiendo trabajar con subconjuntos relevantes sin modificar el dataset original.

In [11]:
df_usd_cop = df[
    (df["base_currency"] == "USD") & 
    (df["target_currency"] == "COP")
]

df_usd_cop

,id,base_currency,target_currency,rate,timestamp
33,34,USD,COP,3600.53,2026-04-18 15:49:37.424755+00:00


### Celda 7: Visualización de la serie temporal (USD → COP)

Esta celda genera una gráfica de la tasa de cambio en el tiempo:

- Crea una figura con `matplotlib`.
- Grafica la serie:
  - Eje X: `timestamp` (fecha)
  - Eje Y: `rate` (tasa de cambio)
- Añade título y etiquetas a los ejes.
- Rota las fechas para mejorar la legibilidad.
- Muestra la gráfica con `plt.show()`.

**Propósito:** visualizar la evolución de la tasa USD → COP a lo largo del tiempo.

**Nota técnica:** esta es una gráfica de serie temporal, útil para identificar tendencias, variaciones y posibles anomalías en los datos.

In [ ]:
plt.figure()
plt.plot(df_usd_cop["timestamp"], df_usd_cop["rate"])
plt.title("USD → COP Exchange Rate")
plt.xlabel("Fecha")
plt.ylabel("Tasa")
plt.xticks(rotation=45)

plt.show()

### Celda 8: Validación de datos para análisis temporal

Esta celda verifica la cantidad de datos disponibles para la serie USD → COP:

- Imprime el número de registros en `df_usd_cop`.
- Muestra el contenido completo del DataFrame filtrado.

**Propósito:** confirmar si existen suficientes datos para realizar un análisis temporal significativo.

**Nota técnica:** si solo hay un registro, no es posible construir una serie temporal útil, ya que no existe histórico para comparar ni identificar tendencias. Esto explica por qué la gráfica anterior no muestra información relevante.

In [13]:
print("Cantidad de registros USD→COP:", len(df_usd_cop))
print(df_usd_cop)

Cantidad de registros USD→COP: 1
    id base_currency target_currency     rate                        timestamp
33  34           USD             COP  3600.53 2026-04-18 15:49:37.424755+00:00
